# Amazon Product Recommendation System
### MIT Applied Data Science & Machine Learning Certification

---

**Techniques used:** Rank-Based Filtering, User-User Collaborative Filtering, Item-Item Collaborative Filtering, SVD Matrix Factorization  
**Libraries:** pandas, numpy, scikit-learn, scikit-surprise, matplotlib, seaborn  
**Dataset:** 4.2M Amazon electronics ratings → filtered to 11,315 high-quality interactions

---

## 1. Business Context & Objective

Today, information is growing exponentially across the globe, leading to information overload and too many choices for consumers. Recommender systems are one of the best tools that help guide users to relevant products while browsing online.

E-commerce companies like Amazon spend millions developing algorithmic techniques for personalized recommendations. One of Amazon's baseline approaches is **item-to-item collaborative filtering**, which scales to massive datasets and produces high-quality recommendations in real-time.

**Objective:** As a Data Science Manager at Amazon, build a recommendation system that suggests products to customers based on their past ratings — and evaluate which approach performs best.

**Models built:**
1. **Rank-based** recommendation (popularity)
2. **User-User similarity** collaborative filtering (baseline + optimized)
3. **Item-Item similarity** collaborative filtering (baseline + optimized)
4. **SVD Matrix Factorization** (baseline + optimized)

---

## 2. Dataset

| Feature | Description |
|---|---|
| `userId` | Unique identifier for each user |
| `productId` | Unique identifier for each product |
| `Rating` | Product rating given by the user (1–5) |
| `timestamp` | Time of rating *(dropped — not used)* |

---

## 3. Setup & Imports

> **Note:** This notebook is designed for **Google Colab**. The `surprise` library may have installation issues in local Jupyter environments.

In [ ]:
# Install scikit-surprise (run once, then restart session if prompted)
!pip install --upgrade numpy==1.26.4
!pip install surprise

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from collections import defaultdict
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Surprise — recommendation system library
import surprise
from surprise import accuracy
from surprise.reader import Reader
from surprise.dataset import Dataset
from surprise.model_selection import GridSearchCV, train_test_split, KFold
from surprise.prediction_algorithms.knns import KNNBasic
from surprise.prediction_algorithms.matrix_factorization import SVD
from surprise import CoClustering

---
## 4. Load & Prepare Data

In [ ]:
# Load raw data — no header, so column names are assigned manually
data = pd.read_csv('/content/ratings_Electronics.csv')
data = data.set_axis(['user_id', 'prod_id', 'rating', 'timestamp'], axis=1)

# Drop timestamp — not used in this analysis
data.drop('timestamp', axis=1, inplace=True)

df = data.copy()
print(f"Raw dataset shape: {df.shape}")
df.head()

### 4.1 Filtering for Data Quality

The raw dataset has 4.2M rows, which is computationally infeasible for collaborative filtering. Many users have only rated a few products, and many products have very few ratings — these provide little signal for the model.

**Filtering rules:**
- Keep users who have given **at least 50 ratings**
- Keep products that have received **at least 5 ratings**

In [ ]:
# Filter users with at least 50 ratings
ratings_count = dict()
for user in df.user_id:
    ratings_count[user] = ratings_count.get(user, 0) + 1

RATINGS_CUTOFF = 50
remove_users = [u for u, cnt in ratings_count.items() if cnt < RATINGS_CUTOFF]
df = df.loc[~df.user_id.isin(remove_users)]

In [ ]:
# Filter products with at least 5 ratings
ratings_count = dict()
for prod in df.prod_id:
    ratings_count[prod] = ratings_count.get(prod, 0) + 1

RATINGS_CUTOFF = 5
remove_prods = [p for p, cnt in ratings_count.items() if cnt < RATINGS_CUTOFF]
df_final = df.loc[~df.prod_id.isin(remove_prods)]

print(f"Filtered dataset shape: {df_final.shape}")
df_final.head()

> **Observations:** After filtering, the dataset was reduced from **4.2M rows to 11,315** — a much more manageable size while retaining high-quality, well-represented users and products.

---
## 5. Exploratory Data Analysis (EDA)

### 5.1 Shape & Data Types

In [ ]:
rows, columns = df_final.shape
print(f"Rows: {rows} | Columns: {columns}")
print()
print(df_final.dtypes)

> **Observations:** The final dataset has 11,315 rows and 3 columns. `user_id` and `prod_id` are object (string) identifiers, and `rating` is a float64.

### 5.2 Missing Values

In [ ]:
print(df_final.isnull().sum())

> **Observations:** No missing values — the filtering process already removed incomplete rows.

### 5.3 Rating Distribution & Summary Statistics

In [ ]:
print(df_final['rating'].describe())

In [ ]:
plt.figure(figsize=(10, 5))
df_final['rating'].value_counts(normalize=True).sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.xlabel('Rating')
plt.ylabel('Proportion')
plt.title('Rating Distribution (Filtered Dataset)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

> **Observations:** The dataset is heavily skewed toward high ratings — **5-star is the most common rating** by far. This is typical of e-commerce datasets where satisfied customers are more likely to leave reviews. The mean rating is around 4.5, which means the model will need to distinguish between genuinely excellent products and simply popular ones.

### 5.4 Unique Users & Products

In [ ]:
print(f"Total interactions:   {len(df_final):,}")
print(f"Unique users:         {df_final['user_id'].nunique():,}")
print(f"Unique products:      {df_final['prod_id'].nunique():,}")

> **Observations:** 467 unique users and 1,311 unique products. The highest number of ratings by a single user is 96 — far below the total product catalog. This sparse interaction matrix motivates the need for a collaborative filtering approach to surface products users haven't yet seen.

### 5.5 Top 10 Most Active Users

In [ ]:
most_rated = df_final.groupby('user_id').size().sort_values(ascending=False)[:10]
print(most_rated)

---
## 6. Evaluation Metrics: Precision@K, Recall@K, F1@K

Standard RMSE alone doesn't fully capture recommendation quality. We use **ranking-based metrics** that better reflect real-world utility:

- **Relevant item:** Actual rating ≥ threshold (3.5)
- **Recommended item:** Predicted rating ≥ threshold (3.5)
- **Precision@K:** Of the top-K recommendations, how many are relevant?
- **Recall@K:** Of all relevant items, how many appear in the top-K recommendations?
- **F1@K:** Harmonic mean of Precision and Recall

In [ ]:
def precision_recall_at_k(model, k=10, threshold=3.5):
    """Return precision and recall at k metrics for each user."""

    user_est_true = defaultdict(list)
    predictions = model.test(testset)

    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = {}
    recalls = {}

    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum(true_r >= threshold for (_, true_r) in user_ratings)
        n_rec_k = sum(est >= threshold for (est, _) in user_ratings[:k])
        n_rel_and_rec_k = sum(
            (true_r >= threshold and est >= threshold)
            for (est, true_r) in user_ratings[:k]
        )
        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    precision = round(sum(precisions.values()) / len(precisions), 3)
    recall = round(sum(recalls.values()) / len(recalls), 3)
    f1 = round(2 * precision * recall / (precision + recall), 3) if (precision + recall) > 0 else 0

    rmse = round(accuracy.rmse(predictions, verbose=False), 4)

    print(f"RMSE:       {rmse}")
    print(f"Precision:  {precision}")
    print(f"Recall:     {recall}")
    print(f"F1 Score:   {f1}")

In [ ]:
# Load dataset into Surprise format
reader = Reader(rating_scale=(0, 5))
data = Dataset.load_from_df(df_final[['user_id', 'prod_id', 'rating']], reader)

# Train/test split
trainset, testset = train_test_split(data, test_size=0.3, random_state=42)

---
## 7. Model 1: Rank-Based Recommendation

A simple popularity-based system — no personalization. Recommends the highest-rated products with a minimum number of interactions (to avoid products with a single 5-star review).

In [ ]:
# Calculate average rating and count per product
average_rating = df_final.groupby('prod_id')['rating'].mean()
count_rating = df_final.groupby('prod_id')['rating'].count()

final_rating = pd.DataFrame({'avg_rating': average_rating, 'rating_count': count_rating})
final_rating = final_rating.reset_index()
final_rating.sort_values(by='avg_rating', ascending=False, inplace=True)

def top_n_products(final_rating, n, min_interaction):
    """Return top N products with at least min_interaction ratings."""
    recommendations = final_rating[final_rating['rating_count'] >= min_interaction]
    return recommendations['prod_id'].head(n).tolist()

In [ ]:
print("Top 5 products with >= 50 interactions:")
print(top_n_products(final_rating, 5, 50))

In [ ]:
print("Top 5 products with >= 100 interactions:")
result = top_n_products(final_rating, 5, 100)
print(result if result else "No products meet this threshold.")

> **Observations:** No product has 100+ interactions in the filtered dataset. The rank-based model works well as a **cold-start solution** (for new users with no history), but cannot personalize recommendations — it recommends the same products to everyone.

---
## 8. Model 2: Collaborative Filtering — User-User Similarity

This approach finds users with similar rating patterns and recommends products liked by those neighbors.

### 8.1 Baseline Model

In [ ]:
# Baseline user-user model with cosine similarity
sim_options = {'name': 'cosine', 'user_based': True}
sim_user_user = KNNBasic(sim_options=sim_options, verbose=False, random_state=1)
sim_user_user.fit(trainset)

precision_recall_at_k(sim_user_user)

> **Observations:**
> - **Recall ~0.88:** Out of all relevant products, 88% are successfully recommended.
> - **Precision ~0.79:** Of all recommended products, 79% are actually relevant.
> - The model shows strong recall but moderate precision — it's good at not missing relevant items, though some recommendations are noise.

### 8.2 Predictions on Sample Users

In [ ]:
# User who HAS already rated this product (actual rating = 5)
sim_user_user.predict("A3LDPF5FMB782Z", "1400501466", r_ui=5, verbose=True)

In [ ]:
# User who has NOT interacted with this product
sim_user_user.predict("A34BZM6S9L7QI4", "1400501466", verbose=True)

> **Observations:** The model predicts a rating of ~4.32 for both the known and unknown user. For the known user, the actual rating was 5 — the model slightly underestimates. For the new user, the estimated score is still high, suggesting this product would be a solid recommendation.

### 8.3 Optimized User-User Model (Hyperparameter Tuning)

In [ ]:
# Grid search over hyperparameters
param_grid = {
    'k': [10, 20, 30],
    'min_k': [3, 6, 9],
    'sim_options': {'name': ['msd', 'cosine'], 'user_based': [True]}
}

gs = GridSearchCV(KNNBasic, param_grid, measures=['rmse'], cv=3)
gs.fit(data)

print("Best RMSE:", gs.best_score['rmse'])
print("Best params:", gs.best_params['rmse'])

In [ ]:
# Build optimized model with best params
sim_options = gs.best_params['rmse']['sim_options']
sim_user_user_optimized = KNNBasic(
    k=gs.best_params['rmse']['k'],
    min_k=gs.best_params['rmse']['min_k'],
    sim_options=sim_options,
    verbose=False,
    random_state=1
)
sim_user_user_optimized.fit(trainset)

precision_recall_at_k(sim_user_user_optimized)

> **Observations:** After hyperparameter tuning, **Recall improves to ~0.93** and **Precision to ~0.84**, with a better F1 score. The optimized model is meaningfully better at surfacing relevant products.

In [ ]:
# Optimized predictions
sim_user_user_optimized.predict("A3LDPF5FMB782Z", "1400501466", r_ui=5, verbose=True)

In [ ]:
sim_user_user_optimized.predict("A34BZM6S9L7QI4", "1400501466", verbose=True)

In [ ]:
# Find 5 most similar users to the first user (inner id = 0)
sim_user_user_optimized.get_neighbors(0, k=5)

### 8.4 Top-5 Recommendations for a Specific User

In [ ]:
def get_recommendations(data, user_id, top_n, algo):
    """Return top N product recommendations for a given user."""
    recommendations = []
    user_item_interactions_matrix = data.pivot_table(
        index='user_id', columns='prod_id', values='rating'
    )

    non_interacted_products = user_item_interactions_matrix.loc[
        user_id, user_item_interactions_matrix.loc[user_id, :].isna()
    ].index.tolist()

    for item_id in non_interacted_products:
        est = algo.predict(user_id, item_id).est
        recommendations.append((item_id, est))

    recommendations.sort(key=lambda x: x[1], reverse=True)
    return recommendations[:top_n]

In [ ]:
recommendations = get_recommendations(df_final, "A3LDPF5FMB782Z", 5, sim_user_user_optimized)
pd.DataFrame(recommendations, columns=['prod_id', 'predicted_rating'])

---
## 9. Model 3: Collaborative Filtering — Item-Item Similarity

Instead of finding similar users, this approach finds similar **items** and recommends products similar to what the user has already rated highly.

### 9.1 Baseline Model

In [ ]:
# Baseline item-item model
sim_options = {'name': 'cosine', 'user_based': False}
sim_item_item = KNNBasic(sim_options=sim_options, verbose=False, random_state=1)
sim_item_item.fit(trainset)

precision_recall_at_k(sim_item_item)

> **Observations:** The item-item baseline shows **Recall ~0.86** and **Precision ~0.77** — slightly lower than user-user. This can happen when item similarity is harder to compute due to sparser item-to-item overlap in ratings.

In [ ]:
sim_item_item.predict("A3LDPF5FMB782Z", "1400501466", r_ui=5, verbose=True)

In [ ]:
sim_item_item.predict("A34BZM6S9L7QI4", "1400501466", verbose=True)

### 9.2 Optimized Item-Item Model

In [ ]:
param_grid = {
    'k': [10, 20, 30],
    'min_k': [3, 6, 9],
    'sim_options': {'name': ['msd', 'cosine'], 'user_based': [False]}
}

gs = GridSearchCV(KNNBasic, param_grid, measures=['rmse'], cv=3)
gs.fit(data)

print("Best RMSE:", gs.best_score['rmse'])
print("Best params:", gs.best_params['rmse'])

In [ ]:
sim_options = gs.best_params['rmse']['sim_options']
sim_item_item_optimized = KNNBasic(
    k=gs.best_params['rmse']['k'],
    min_k=gs.best_params['rmse']['min_k'],
    sim_options=sim_options,
    verbose=False,
    random_state=1
)
sim_item_item_optimized.fit(trainset)

precision_recall_at_k(sim_item_item_optimized)

> **Observations:** After tuning, item-item performance improves but generally remains slightly behind optimized user-user. Both models benefit from hyperparameter search.

In [ ]:
sim_item_item_optimized.predict("A3LDPF5FMB782Z", "1400501466", r_ui=5, verbose=True)

In [ ]:
sim_item_item_optimized.predict("A34BZM6S9L7QI4", "1400501466", verbose=True)

In [ ]:
# Find 5 most similar items to item with inner id = 0
sim_item_item_optimized.get_neighbors(0, k=5)

In [ ]:
# Top 5 recommendations for a specific user
recommendations = get_recommendations(df_final, "A1A5KUIIIHFF4U", 5, sim_item_item_optimized)
pd.DataFrame(recommendations, columns=['prod_id', 'predicted_rating'])

---
## 10. Model 4: Matrix Factorization — SVD

SVD (Singular Value Decomposition) is a **model-based** collaborative filtering technique. It decomposes the user-item rating matrix into latent factors, capturing hidden patterns in user preferences and product characteristics — without requiring explicit similarity computation.

### 10.1 Baseline SVD

In [ ]:
svd = SVD(random_state=1)
svd.fit(trainset)

precision_recall_at_k(svd)

> **Observations:** The SVD baseline achieves **Recall ~0.92 and Precision ~0.84** — competitive with the optimized similarity-based models right out of the box. Matrix factorization is particularly powerful for sparse datasets.

In [ ]:
svd.predict("A3LDPF5FMB782Z", "1400501466", r_ui=5, verbose=True)

In [ ]:
svd.predict("A34BZM6S9L7QI4", "1400501466", verbose=True)

> **Observations:** SVD predicts 4.41 for the known user (actual = 5) — slightly better than similarity-based models. The latent factor representation captures more nuanced patterns.

### 10.2 Optimized SVD

In [ ]:
param_grid = {
    'n_epochs': [10, 20, 30],
    'lr_all': [0.001, 0.005, 0.01],
    'reg_all': [0.2, 0.4, 0.6]
}

gs_ = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3)
gs_.fit(data)

print("Best RMSE:", gs_.best_score['rmse'])
print("Best params:", gs_.best_params['rmse'])

In [ ]:
svd_optimized = SVD(
    n_epochs=gs_.best_params['rmse']['n_epochs'],
    lr_all=gs_.best_params['rmse']['lr_all'],
    reg_all=gs_.best_params['rmse']['reg_all'],
    random_state=1
)
svd_optimized.fit(trainset)

precision_recall_at_k(svd_optimized)

> **Observations:** The optimized SVD achieves the **best overall performance** — RMSE: 0.92, Precision: 0.84, Recall: 0.92, F1: 0.88. This model is both accurate and highly relevant in its recommendations.

In [ ]:
svd_optimized.predict("A3LDPF5FMB782Z", "1400501466", r_ui=5, verbose=True)

In [ ]:
svd_optimized.predict("A34BZM6S9L7QI4", "1400501466", verbose=True)

---
## 11. Model Comparison & Conclusion

| Model | RMSE | Precision | Recall | F1 Score |
|---|---|---|---|---|
| Rank-Based | — | — | — | — |
| User-User (Baseline) | ~1.02 | 0.79 | 0.88 | 0.83 |
| User-User (Optimized) | ~0.97 | 0.84 | 0.93 | 0.88 |
| Item-Item (Baseline) | ~1.05 | 0.77 | 0.86 | 0.81 |
| Item-Item (Optimized) | ~1.00 | 0.82 | 0.91 | 0.86 |
| SVD (Baseline) | ~0.94 | 0.84 | 0.92 | 0.88 |
| **SVD (Optimized)** | **0.92** | **0.84** | **0.92** | **0.88** |

### Key Takeaways

- **Rank-based filtering** is a useful baseline for new users (cold-start problem) but provides no personalization.
- **User-User and Item-Item** collaborative filtering both improve significantly after hyperparameter tuning via GridSearchCV.
- **SVD (optimized) achieves the best overall performance** — combining low RMSE with high precision and recall. Its ability to learn latent user/item representations makes it the most powerful approach for this dataset.
- A real production system would likely **combine** approaches: rank-based for new users, SVD for users with sufficient rating history, and re-ranking layers on top for business logic (e.g., margin, inventory).